# 08 LGD and EAD Preparation

Milestone 6 prepares Loss Given Default and Exposure at Default inputs for later Expected Loss calculations. This notebook creates reliable LGD/EAD datasets and preliminary Expected Loss fields, but it does not produce the final analytical report.

## 1. Load Cleaned Data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except ModuleNotFoundError:
    SEABORN_AVAILABLE = False
    print("seaborn is not installed. Plots will use matplotlib only.")

In [ ]:
project_root = Path("..").resolve()
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "reports" / "figures"

processed_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

loans_clean_path = processed_dir / "loans_clean_v1.csv"
loans_with_pd_path = processed_dir / "loans_with_pd_v1.csv"
final_pd_test_predictions_path = processed_dir / "final_pd_test_predictions_v1.csv"

loans = pd.read_csv(loans_clean_path, low_memory=False)
final_pd_test_predictions = pd.read_csv(final_pd_test_predictions_path, low_memory=False)

if "default_flag" not in loans.columns:
    raise ValueError("default_flag is required but was not found in loans_clean_v1.csv.")

print(f"loans_clean_v1 shape: {loans.shape}")
print(f"final_pd_test_predictions_v1 shape: {final_pd_test_predictions.shape}")
print("default_flag distribution:")
print(loans["default_flag"].value_counts(dropna=False).sort_index())

## 2. Inspect Required Fields

In [ ]:
fields_to_check = [
    "funded_amnt", "loan_amnt", "recoveries", "collection_recovery_fee",
    "total_pymnt", "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "last_pymnt_amnt", "out_prncp", "grade", "sub_grade", "term", "purpose", "issue_d",
]
essential_fields = ["funded_amnt", "recoveries", "default_flag"]

field_check = pd.DataFrame({
    "field": fields_to_check,
    "exists": [field in loans.columns for field in fields_to_check],
})
display(field_check)

missing_essential_fields = [field for field in essential_fields if field not in loans.columns]
if missing_essential_fields:
    raise ValueError(f"Missing essential LGD/EAD fields: {missing_essential_fields}")

print("All essential fields are present.")

## 3. Create EAD Proxy

For Milestone 6, Exposure at Default is approximated using `funded_amnt`. This is a simple first proxy and can be improved later.

In [ ]:
loans_lgd_ead = loans.copy()
loans_lgd_ead["funded_amnt"] = pd.to_numeric(loans_lgd_ead["funded_amnt"], errors="coerce")
loans_lgd_ead["ead_proxy"] = loans_lgd_ead["funded_amnt"]

loans_lgd_ead["missing_funded_amnt_flag"] = loans_lgd_ead["funded_amnt"].isna()
loans_lgd_ead["non_positive_funded_amnt_flag"] = loans_lgd_ead["funded_amnt"].le(0).fillna(False)
loans_lgd_ead["valid_ead_proxy_flag"] = (
    loans_lgd_ead["funded_amnt"].notna()
    & loans_lgd_ead["funded_amnt"].gt(0)
)

print(f"Missing funded_amnt rows: {loans_lgd_ead['missing_funded_amnt_flag'].sum():,}")
print(f"Zero or negative funded_amnt rows: {loans_lgd_ead['non_positive_funded_amnt_flag'].sum():,}")
print("EAD proxy summary:")
display(loans_lgd_ead["ead_proxy"].describe())

## 4. Create LGD for Defaulted Loans

LGD is calculated only for defaulted loans where `default_flag == 1`. Recoveries are filled with zero only after creating a missing-recoveries flag.

In [ ]:
defaulted_loans = loans_lgd_ead[loans_lgd_ead["default_flag"] == 1].copy()
defaulted_loans["recoveries"] = pd.to_numeric(defaulted_loans["recoveries"], errors="coerce")

defaulted_loans["missing_recoveries_flag"] = defaulted_loans["recoveries"].isna()
defaulted_loans["recoveries_filled"] = defaulted_loans["recoveries"].fillna(0)

defaulted_loans["recovery_rate"] = np.where(
    defaulted_loans["valid_ead_proxy_flag"],
    defaulted_loans["recoveries_filled"] / defaulted_loans["ead_proxy"],
    np.nan,
)
defaulted_loans["lgd_raw"] = 1 - defaulted_loans["recovery_rate"]
defaulted_loans["lgd"] = defaulted_loans["lgd_raw"].clip(lower=0, upper=1)
defaulted_loans["lgd_clipped_flag"] = (
    defaulted_loans["lgd_raw"].lt(0)
    | defaulted_loans["lgd_raw"].gt(1)
)

print(f"Defaulted loan rows: {len(defaulted_loans):,}")
print(f"Missing recoveries rows: {defaulted_loans['missing_recoveries_flag'].sum():,}")
print(f"Invalid EAD rows among defaults: {(~defaulted_loans['valid_ead_proxy_flag']).sum():,}")
print(f"Clipped LGD rows: {defaulted_loans['lgd_clipped_flag'].sum():,}")
display(defaulted_loans[["funded_amnt", "ead_proxy", "recoveries", "recoveries_filled", "recovery_rate", "lgd_raw", "lgd"]].head())

## 5. Summary Tables

In [ ]:
def share(series):
    """Return the mean of a boolean series as a share."""
    return series.mean()


lgd_summary_stats = defaulted_loans["lgd"].describe().to_frame("lgd")
recovery_rate_summary_stats = defaulted_loans["recovery_rate"].describe().to_frame("recovery_rate")

lgd_quality_summary = pd.DataFrame({
    "metric": [
        "zero_recovery_share",
        "lgd_equal_1_share",
        "lgd_equal_0_share",
        "clipped_lgd_share",
        "missing_recoveries_share",
        "invalid_ead_share",
    ],
    "value": [
        share(defaulted_loans["recoveries_filled"].eq(0)),
        share(defaulted_loans["lgd"].eq(1)),
        share(defaulted_loans["lgd"].eq(0)),
        share(defaulted_loans["lgd_clipped_flag"].fillna(False)),
        share(defaulted_loans["missing_recoveries_flag"]),
        share(~defaulted_loans["valid_ead_proxy_flag"]),
    ],
})

display(lgd_summary_stats)
display(recovery_rate_summary_stats)
display(lgd_quality_summary)

In [ ]:
def segment_lgd_summary(data, segment_column):
    """Create LGD segment summary if the segment column exists."""
    if segment_column not in data.columns:
        print(f"Skipping {segment_column}: column not available.")
        return None

    summary = (
        data.groupby(segment_column, dropna=False)
        .agg(
            count=("lgd", "size"),
            mean_lgd=("lgd", "mean"),
            median_lgd=("lgd", "median"),
            mean_recovery_rate=("recovery_rate", "mean"),
            mean_ead=("ead_proxy", "mean"),
        )
        .reset_index()
        .sort_values("mean_lgd", ascending=False)
    )
    return summary


segment_summaries = {}
for segment_column in ["grade", "sub_grade", "term", "purpose"]:
    segment_summary = segment_lgd_summary(defaulted_loans, segment_column)
    if segment_summary is not None:
        segment_summaries[segment_column] = segment_summary
        print(f"LGD by {segment_column}")
        display(segment_summary.head(15))

if "issue_d" in defaulted_loans.columns:
    defaulted_loans["issue_date"] = pd.to_datetime(defaulted_loans["issue_d"], format="%b-%Y", errors="coerce")
    defaulted_loans["issue_year"] = defaulted_loans["issue_date"].dt.year
    issue_year_summary = segment_lgd_summary(defaulted_loans, "issue_year")
    segment_summaries["issue_year"] = issue_year_summary
    print("LGD by issue_year")
    display(issue_year_summary)
else:
    print("issue_d is not available, so LGD by issue year cannot be created.")

## 6. Plots

In [ ]:
def save_histogram(series, title, xlabel, output_path, bins=40):
    plt.figure(figsize=(8, 5))
    if SEABORN_AVAILABLE:
        sns.histplot(series.dropna(), bins=bins, kde=False)
    else:
        plt.hist(series.dropna(), bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Number of Loans")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()


def save_bar_chart(data, x_col, y_col, title, xlabel, ylabel, output_path):
    plt.figure(figsize=(9, 5))
    if SEABORN_AVAILABLE:
        sns.barplot(data=data, x=x_col, y=y_col, color="steelblue")
    else:
        plt.bar(data[x_col].astype(str), data[y_col])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()


save_histogram(defaulted_loans["lgd"], "LGD Distribution", "LGD", figures_dir / "lgd_histogram_v1.png")
save_histogram(defaulted_loans["recovery_rate"], "Recovery Rate Distribution", "Recovery Rate", figures_dir / "recovery_rate_histogram_v1.png")

if "grade" in segment_summaries:
    grade_plot = segment_summaries["grade"].sort_values("grade")
    save_bar_chart(grade_plot, "grade", "mean_lgd", "Average LGD by Grade", "Grade", "Average LGD", figures_dir / "avg_lgd_by_grade_v1.png")

if "term" in segment_summaries:
    term_plot = segment_summaries["term"].sort_values("term")
    save_bar_chart(term_plot, "term", "mean_lgd", "Average LGD by Term", "Term", "Average LGD", figures_dir / "avg_lgd_by_term_v1.png")

if "purpose" in segment_summaries:
    top_purposes = segment_summaries["purpose"].sort_values("count", ascending=False).head(10)
    save_bar_chart(top_purposes, "purpose", "mean_lgd", "Average LGD by Top Purposes", "Purpose", "Average LGD", figures_dir / "avg_lgd_by_top_purposes_v1.png")

if "issue_year" in segment_summaries:
    issue_year_plot = segment_summaries["issue_year"].sort_values("issue_year")
    save_bar_chart(issue_year_plot, "issue_year", "mean_lgd", "Average LGD by Issue Year", "Issue Year", "Average LGD", figures_dir / "avg_lgd_by_issue_year_v1.png")

## 7. Portfolio-Average LGD Baseline

In [ ]:
portfolio_avg_lgd = defaulted_loans["lgd"].mean()

loans_lgd_ead["lgd_estimate"] = portfolio_avg_lgd
loans_lgd_ead["lgd_estimate_method"] = "portfolio_average_defaulted_loans"

print(f"Portfolio average LGD among defaulted loans: {portfolio_avg_lgd:.4f}")

## 8. Create All-Loan LGD/EAD Dataset

In [ ]:
base_lgd_ead_columns = [
    "id", "loan_status", "default_flag", "funded_amnt", "ead_proxy",
    "lgd_estimate", "lgd_estimate_method", "grade", "sub_grade", "term", "purpose", "issue_d",
]
lgd_ead_columns = [column for column in base_lgd_ead_columns if column in loans_lgd_ead.columns]
lgd_ead_dataset = loans_lgd_ead[lgd_ead_columns].copy()

lgd_ead_dataset_path = processed_dir / "lgd_ead_dataset_v1.csv"
lgd_ead_dataset.to_csv(lgd_ead_dataset_path, index=False)

print(f"Saved LGD/EAD dataset to: {lgd_ead_dataset_path}")
print(f"Shape: {lgd_ead_dataset.shape}")

## 9. Save Defaulted-Loan LGD Dataset

In [ ]:
base_defaulted_columns = [
    "id", "loan_status", "default_flag", "funded_amnt", "ead_proxy",
    "recoveries", "recoveries_filled", "missing_recoveries_flag",
    "recovery_rate", "lgd_raw", "lgd", "lgd_clipped_flag",
    "grade", "sub_grade", "term", "purpose", "issue_d",
]
defaulted_columns = [column for column in base_defaulted_columns if column in defaulted_loans.columns]
defaulted_loans_lgd = defaulted_loans[defaulted_columns].copy()

defaulted_loans_lgd_path = processed_dir / "defaulted_loans_lgd_v1.csv"
defaulted_loans_lgd.to_csv(defaulted_loans_lgd_path, index=False)

print(f"Saved defaulted-loan LGD dataset to: {defaulted_loans_lgd_path}")
print(f"Shape: {defaulted_loans_lgd.shape}")

## 10. Combine with Final PD Outputs

In [ ]:
loans_with_pd = pd.read_csv(loans_with_pd_path, low_memory=False)

pd_column_candidates = [
    "pd_final", "pd_pred", "pd", "default_probability",
    "predicted_pd", "pd_l2", "final_pd",
]
pd_column = next((column for column in pd_column_candidates if column in loans_with_pd.columns), None)

if pd_column is None:
    print("Available columns in loans_with_pd_v1.csv:")
    print(list(loans_with_pd.columns))
    raise ValueError("No obvious PD column found. Please identify the PD column to use.")

loans_with_pd_lgd_ead = loans_with_pd.copy()
loans_with_pd_lgd_ead["funded_amnt"] = pd.to_numeric(loans_with_pd_lgd_ead["funded_amnt"], errors="coerce")
loans_with_pd_lgd_ead["ead_proxy"] = loans_with_pd_lgd_ead["funded_amnt"]
loans_with_pd_lgd_ead["lgd_estimate"] = portfolio_avg_lgd
loans_with_pd_lgd_ead["lgd_estimate_method"] = "portfolio_average_defaulted_loans"
loans_with_pd_lgd_ead["expected_loss_prelim"] = (
    loans_with_pd_lgd_ead[pd_column]
    * loans_with_pd_lgd_ead["lgd_estimate"]
    * loans_with_pd_lgd_ead["ead_proxy"]
)

pd_valid = loans_with_pd_lgd_ead[pd_column].between(0, 1, inclusive="both")
lgd_valid = loans_with_pd_lgd_ead["lgd_estimate"].between(0, 1, inclusive="both")
ead_valid = loans_with_pd_lgd_ead["ead_proxy"].ge(0).fillna(False)
el_non_negative = loans_with_pd_lgd_ead["expected_loss_prelim"].ge(0).fillna(False)
el_less_than_ead = (
    loans_with_pd_lgd_ead["expected_loss_prelim"]
    <= loans_with_pd_lgd_ead["ead_proxy"] + 1e-8
).fillna(False)

validation_summary = pd.DataFrame({
    "check": [
        "PD between 0 and 1",
        "LGD estimate between 0 and 1",
        "EAD non-negative",
        "Expected loss non-negative",
        "Expected loss <= EAD",
        "Row count preserved versus loans_with_pd",
    ],
    "passed": [
        pd_valid.all(),
        lgd_valid.all(),
        ead_valid.all(),
        el_non_negative.all(),
        el_less_than_ead.all(),
        len(loans_with_pd_lgd_ead) == len(loans_with_pd),
    ],
})
display(validation_summary)

if not validation_summary["passed"].all():
    raise ValueError("One or more PD/LGD/EAD validation checks failed. Review validation_summary above.")

loans_with_pd_lgd_ead_path = processed_dir / "loans_with_pd_lgd_ead_v1.csv"
loans_with_pd_lgd_ead.to_csv(loans_with_pd_lgd_ead_path, index=False)

print(f"Detected PD column: {pd_column}")
print(f"Saved combined PD/LGD/EAD dataset to: {loans_with_pd_lgd_ead_path}")
print(f"Shape: {loans_with_pd_lgd_ead.shape}")

## 11. Output Checks

In [ ]:
output_summary = pd.DataFrame({
    "output": [
        "defaulted_loans_lgd_v1.csv",
        "lgd_ead_dataset_v1.csv",
        "loans_with_pd_lgd_ead_v1.csv",
    ],
    "path": [
        str(defaulted_loans_lgd_path),
        str(lgd_ead_dataset_path),
        str(loans_with_pd_lgd_ead_path),
    ],
    "shape": [
        defaulted_loans_lgd.shape,
        lgd_ead_dataset.shape,
        loans_with_pd_lgd_ead.shape,
    ],
    "exists": [
        defaulted_loans_lgd_path.exists(),
        lgd_ead_dataset_path.exists(),
        loans_with_pd_lgd_ead_path.exists(),
    ],
})

display(output_summary)
print(f"Portfolio average LGD: {portfolio_avg_lgd:.4f}")
print(f"Mean EAD: {loans_with_pd_lgd_ead['ead_proxy'].mean():,.2f}")
print(f"Mean preliminary expected loss: {loans_with_pd_lgd_ead['expected_loss_prelim'].mean():,.2f}")
print(f"Total preliminary expected loss: {loans_with_pd_lgd_ead['expected_loss_prelim'].sum():,.2f}")